# 04 — Gold Star Schema

Consumption-ready star schema for dashboards, Genie, and ML:
5 dimensions, 2 facts, 3 pre-aggregated views.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Performance Efficiency** | Liquid clustering on `fact_pitches(season, official_date, pitcher_sk)` is tuned *exactly* to the three filter+group-by predicates the Lakeview dashboard and the pitcher leaderboard emit. This is the table people will actually hit hard. |
| **Usability** | `dim_* + fact_*` is the shape AI/BI visualization suggestions key off of. With PKs and FKs wired up, AI/BI can *infer the join graph automatically* — the gold layer buys you autopilot for the consumption experience. |
| **Operational Excellence** | Dimensions are Type-1 (overwrite) and rebuilt every run from silver — no SCD machinery to maintain. When requirements demand Type-2, we'd reach for DLT + APPLY CHANGES, not hand-rolled Python. |
| **Cost Optimization** | Pre-aggregated views (`v_pitcher_leaderboard`, `v_pitch_mix_by_type`, `v_team_pitching_summary`) mean the dashboard's KPI tiles scan kilobytes instead of re-aggregating 700k rows every page load. Same dashboard, ~10× cheaper at scale. |


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Silver: {S}")
print(f"Gold:   {G}")


Silver: alexander_booth.mlb_gumbo_waf_silver
Gold:   alexander_booth.mlb_gumbo_waf_gold


In [2]:
def add_pk(table, name, cols):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")


def add_fk(table, name, cols, ref_table, ref_cols):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)})")

print("Helpers loaded.")


Helpers loaded.


## dim_date

In [3]:
spark.sql(f"DROP TABLE IF EXISTS {G}.dim_date")
spark.sql(f"""
    CREATE TABLE {G}.dim_date AS
    WITH d AS (
        SELECT EXPLODE(SEQUENCE(DATE'2020-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date_value
    )
    SELECT
        CAST(DATE_FORMAT(date_value, 'yyyyMMdd') AS INT) AS date_sk,
        date_value,
        YEAR(date_value)       AS year,
        MONTH(date_value)      AS month,
        DAY(date_value)        AS day,
        QUARTER(date_value)    AS quarter,
        WEEKOFYEAR(date_value) AS week_of_year,
        DAYOFWEEK(date_value)  AS day_of_week,
        DATE_FORMAT(date_value, 'MMMM') AS month_name,
        DATE_FORMAT(date_value, 'EEEE') AS day_name,
        CASE WHEN DAYOFWEEK(date_value) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend
    FROM d
""")
add_pk(f"{G}.dim_date", "dim_date_pk", ["date_sk"])
print(f"dim_date: {spark.table(f'{G}.dim_date').count():,} rows")


dim_date: 4,018 rows


## dim_team

In [4]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_team AS
    WITH all_teams AS (
        SELECT home_team_id AS team_id, home_team_name AS team_name FROM {S}.game_data
        UNION
        SELECT away_team_id, away_team_name FROM {S}.game_data
    )
    SELECT
        MD5(CAST(team_id AS STRING)) AS team_sk,
        team_id,
        team_name
    FROM all_teams
    WHERE team_id IS NOT NULL
    GROUP BY team_id, team_name
""")
add_pk(f"{G}.dim_team", "dim_team_pk", ["team_sk"])
print(f"dim_team: {spark.table(f'{G}.dim_team').count():,} rows")


dim_team: 30 rows


## dim_venue

In [5]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_venue AS
    SELECT
        MD5(CAST(venue_id AS STRING)) AS venue_sk,
        venue_id,
        ANY_VALUE(venue_name) AS venue_name
    FROM {S}.game_data
    WHERE venue_id IS NOT NULL
    GROUP BY venue_id
""")
add_pk(f"{G}.dim_venue", "dim_venue_pk", ["venue_sk"])
print(f"dim_venue: {spark.table(f'{G}.dim_venue').count():,} rows")


dim_venue: 33 rows


## dim_pitcher, dim_batter

In [6]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_pitcher AS
    SELECT
        MD5(CAST(pitcher_id AS STRING)) AS pitcher_sk,
        pitcher_id,
        ANY_VALUE(pitcher_name) AS pitcher_name,
        ANY_VALUE(pitcher_hand) AS pitcher_hand
    FROM {S}.pitch_data
    WHERE pitcher_id IS NOT NULL
    GROUP BY pitcher_id
""")
add_pk(f"{G}.dim_pitcher", "dim_pitcher_pk", ["pitcher_sk"])

spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_batter AS
    SELECT
        MD5(CAST(batter_id AS STRING)) AS batter_sk,
        batter_id,
        ANY_VALUE(batter_name) AS batter_name,
        ANY_VALUE(batter_side) AS batter_side
    FROM {S}.pitch_data
    WHERE batter_id IS NOT NULL
    GROUP BY batter_id
""")
add_pk(f"{G}.dim_batter", "dim_batter_pk", ["batter_sk"])

print(f"dim_pitcher: {spark.table(f'{G}.dim_pitcher').count():,} rows")
print(f"dim_batter:  {spark.table(f'{G}.dim_batter').count():,} rows")


dim_pitcher: 873 rows


dim_batter:  674 rows


## fact_games

In [7]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_games (
        game_sk           STRING  NOT NULL,
        game_pk           INT,
        date_sk           INT,
        home_team_sk      STRING,
        away_team_sk      STRING,
        venue_sk          STRING,
        season            INT,
        official_date     DATE,
        weather_temp_f    INT,
        attendance        INT,
        flags_no_hitter   BOOLEAN,
        flags_perfect_game BOOLEAN
    )
    CLUSTER BY (season, official_date)
    COMMENT 'Fact table at game grain. One row per MLB game.'
""")

spark.sql(f"""
    INSERT OVERWRITE {G}.fact_games
    SELECT
        g.game_sk,
        g.game_pk,
        CAST(DATE_FORMAT(g.official_date, 'yyyyMMdd') AS INT) AS date_sk,
        MD5(CAST(g.home_team_id AS STRING)) AS home_team_sk,
        MD5(CAST(g.away_team_id AS STRING)) AS away_team_sk,
        MD5(CAST(g.venue_id    AS STRING)) AS venue_sk,
        g.season,
        g.official_date,
        g.weather_temp_f,
        g.attendance,
        g.flags_no_hitter,
        g.flags_perfect_game
    FROM {S}.game_data g
""")

add_pk(f"{G}.fact_games", "fact_games_pk", ["game_sk"])
add_fk(f"{G}.fact_games", "fg_date_fk",     ["date_sk"],      f"{G}.dim_date",   ["date_sk"])
add_fk(f"{G}.fact_games", "fg_home_team_fk", ["home_team_sk"], f"{G}.dim_team",   ["team_sk"])
add_fk(f"{G}.fact_games", "fg_away_team_fk", ["away_team_sk"], f"{G}.dim_team",   ["team_sk"])
add_fk(f"{G}.fact_games", "fg_venue_fk",    ["venue_sk"],     f"{G}.dim_venue",  ["venue_sk"])

print(f"fact_games: {spark.table(f'{G}.fact_games').count():,} rows")


fact_games: 2,477 rows


## fact_pitches

In [8]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_pitches (
        pitch_sk               STRING NOT NULL,
        game_sk                STRING,
        date_sk                INT,
        pitcher_sk             STRING,
        batter_sk              STRING,
        season                 INT,
        official_date          DATE,
        inning                 INT,
        half_inning            STRING,
        pitch_type_code        STRING,
        pitch_type_description STRING,
        call_code              STRING,
        call_description       STRING,
        is_strike              BOOLEAN,
        is_in_play             BOOLEAN,
        start_speed_mph        DOUBLE,
        spin_rate_rpm          INT,
        plate_x                DOUBLE,
        plate_z                DOUBLE,
        sz_top                 DOUBLE,
        sz_bot                 DOUBLE,
        break_vertical_induced DOUBLE,
        break_horizontal       DOUBLE,
        extension_ft           DOUBLE,
        batter_side            STRING,
        pitcher_hand           STRING
    )
    CLUSTER BY (season, official_date, pitcher_sk)
    COMMENT 'Fact table at pitch grain. One row per pitch event across the season.'
""")

spark.sql(f"""
    INSERT OVERWRITE {G}.fact_pitches
    SELECT
        p.pitch_sk,
        p.game_sk,
        CAST(DATE_FORMAT(p.official_date, 'yyyyMMdd') AS INT) AS date_sk,
        MD5(CAST(p.pitcher_id AS STRING)) AS pitcher_sk,
        MD5(CAST(p.batter_id  AS STRING)) AS batter_sk,
        p.season,
        p.official_date,
        p.inning,
        p.half_inning,
        p.pitch_type_code,
        p.pitch_type_description,
        p.call_code,
        p.call_description,
        p.is_strike,
        p.is_in_play,
        p.start_speed_mph,
        p.spin_rate_rpm,
        p.plate_x,
        p.plate_z,
        p.sz_top,
        p.sz_bot,
        p.break_vertical_induced,
        p.break_horizontal,
        p.extension_ft,
        p.batter_side,
        p.pitcher_hand
    FROM {S}.pitch_data p
""")

add_pk(f"{G}.fact_pitches", "fact_pitches_pk", ["pitch_sk"])
add_fk(f"{G}.fact_pitches", "fp_game_fk",    ["game_sk"],    f"{G}.fact_games",   ["game_sk"])
add_fk(f"{G}.fact_pitches", "fp_date_fk",    ["date_sk"],    f"{G}.dim_date",     ["date_sk"])
add_fk(f"{G}.fact_pitches", "fp_pitcher_fk", ["pitcher_sk"], f"{G}.dim_pitcher",  ["pitcher_sk"])
add_fk(f"{G}.fact_pitches", "fp_batter_fk",  ["batter_sk"],  f"{G}.dim_batter",   ["batter_sk"])

print(f"fact_pitches: {spark.table(f'{G}.fact_pitches').count():,} rows")


fact_pitches: 724,180 rows


## Pre-aggregated views

In [9]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_pitcher_leaderboard AS
    SELECT
        dp.pitcher_name,
        dp.pitcher_hand,
        COUNT(*) AS total_pitches,
        ROUND(AVG(fp.start_speed_mph), 2) AS avg_velocity_mph,
        ROUND(AVG(fp.spin_rate_rpm), 1)   AS avg_spin_rpm,
        ROUND(100.0 * SUM(CASE WHEN fp.is_strike THEN 1 ELSE 0 END) / COUNT(*), 2) AS strike_pct
    FROM {G}.fact_pitches fp
    JOIN {G}.dim_pitcher dp USING (pitcher_sk)
    GROUP BY dp.pitcher_name, dp.pitcher_hand
    HAVING COUNT(*) > 200
""")
print("Created: v_pitcher_leaderboard")

spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_pitch_mix_by_type AS
    SELECT
        season,
        pitch_type_description,
        COUNT(*) AS n_pitches,
        ROUND(AVG(start_speed_mph), 2) AS avg_velocity_mph,
        ROUND(AVG(spin_rate_rpm), 1)   AS avg_spin_rpm,
        ROUND(AVG(break_vertical_induced), 2) AS avg_ivb,
        ROUND(AVG(break_horizontal), 2)       AS avg_horz_break,
        ROUND(100.0 * SUM(CASE WHEN is_strike THEN 1 ELSE 0 END) / COUNT(*), 2) AS strike_pct
    FROM {G}.fact_pitches
    WHERE pitch_type_description IS NOT NULL
    GROUP BY season, pitch_type_description
""")
print("Created: v_pitch_mix_by_type")

spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_team_pitching_summary AS
    SELECT
        fp.season,
        dt_home.team_name AS team_name,
        COUNT(DISTINCT fg.game_sk) AS games,
        COUNT(*)                    AS pitches,
        ROUND(AVG(fp.start_speed_mph), 2) AS avg_velocity_mph,
        ROUND(100.0 * SUM(CASE WHEN fp.is_strike THEN 1 ELSE 0 END) / COUNT(*), 2) AS strike_pct
    FROM {G}.fact_pitches fp
    JOIN {G}.fact_games fg ON fp.game_sk = fg.game_sk
    JOIN {G}.dim_team dt_home ON fg.home_team_sk = dt_home.team_sk
    GROUP BY fp.season, dt_home.team_name
""")
print("Created: v_team_pitching_summary")


Created: v_pitcher_leaderboard


Created: v_pitch_mix_by_type


Created: v_team_pitching_summary


In [10]:
print("=" * 60)
print("GOLD LAYER SUMMARY")
print("=" * 60)
for t in ["dim_date", "dim_team", "dim_venue", "dim_pitcher", "dim_batter", "fact_games", "fact_pitches"]:
    n = spark.table(f"{G}.{t}").count()
    print(f"  {G}.{t}: {n:,} rows")
print("\nViews: v_pitcher_leaderboard, v_pitch_mix_by_type, v_team_pitching_summary")


GOLD LAYER SUMMARY


  alexander_booth.mlb_gumbo_waf_gold.dim_date: 4,018 rows


  alexander_booth.mlb_gumbo_waf_gold.dim_team: 30 rows


  alexander_booth.mlb_gumbo_waf_gold.dim_venue: 33 rows


  alexander_booth.mlb_gumbo_waf_gold.dim_pitcher: 873 rows


  alexander_booth.mlb_gumbo_waf_gold.dim_batter: 674 rows


  alexander_booth.mlb_gumbo_waf_gold.fact_games: 2,477 rows


  alexander_booth.mlb_gumbo_waf_gold.fact_pitches: 724,180 rows

Views: v_pitcher_leaderboard, v_pitch_mix_by_type, v_team_pitching_summary


> ## WAF takeaway — what to say on this slide
>
> - The **star schema shape is the product.** Everything downstream —
>   dashboards, Genie, ML features — is a consumer of this shape.
> - We chose **views, not materialized views**, for the pre-aggregations
>   because the underlying gold tables are already clustered on the right
>   keys — the marginal benefit of MVs for this demo isn't worth the
>   refresh cost. In prod, promote to MVs once query latency becomes the
>   bottleneck, not before. That's WAF: **don't over-engineer**.
